In [ ]:
!pip install transformers accelerate einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 103.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 89.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 56.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 113.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5

In [ ]:
import random
import json
from typing import List, Dict
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# ----------------------------
# 1. Load Qwen Model (you may need to install and authenticate)
# ----------------------------

model_name = "Qwen/Qwen1.5-7B-Chat"  # adjust if needed
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

# ----------------------------
# 2. Define Contradiction Scenarios
# ----------------------------

CONTRADICTION_TYPES = {
    "contextual": "User's preference changes temporarily based on situation (e.g., health, mood).",
    "trade_off": "User wants mutually exclusive things (e.g., short but detailed responses).",
    "topic_specific": "User preference depends on topic or task.",
    "temporal": "User's preference evolves over time (permanent change).",
    "ambiguous": "User expresses preferences vaguely or with hedging language.",
    "meta_preference": "User sets a rule for how their preferences should be handled."
}

EXAMPLE_PROMPTS = {
    "contextual": "User: I love spicy food.\nAgent: Got it, you love spicy food.\nUser (later): I'm sick today, nothing spicy please.",
    "trade_off": "User: Please keep responses short.\nAgent: Sure.\nUser (later): Why is your answer so vague? I want more details.",
    "topic_specific": "User: I prefer short answers.\nAgent: Understood.\nUser (later, different topic): Give me a thorough explanation.",
    "temporal": "User: I like horror movies.\nAgent: Got it.\nUser (weeks later): I don’t like horror anymore, it stresses me out.",
    "ambiguous": "User: I kinda like when things are concise...\nAgent: OK, I’ll be brief.\nUser (later): You left out too much info.",
    "meta_preference": "User: By default, keep it short, but explain more if I ask follow-up questions."
}

# ----------------------------
# 3. Generate Response Pairs Using Qwen
# ----------------------------

def generate_dpo_pair(prompt: str, contradiction_type: str) -> Dict:
    instruction = (
        "Given this dialogue with a user contradiction (" + contradiction_type + "), "
        "generate:\nPreferred: A helpful agent response that addresses or clarifies the contradiction.\n"
        "Dispreferred: A response that ignores or mishandles the contradiction.\n"
        "Respond naturally and clearly label each response.\n\n"
        f"Dialogue:\n{prompt}\n\nPreferred:"
    )

    output = generator(instruction, max_new_tokens=250, do_sample=True, temperature=0.7)[0]['generated_text']

    # Extract preferred/dispreferred responses
    try:
        preferred = output.split("Preferred:")[1].split("Dispreferred:")[0].strip()
        dispreferred = output.split("Dispreferred:")[1].strip()
    except Exception as e:
        preferred = "ERROR_PARSING"
        dispreferred = "ERROR_PARSING"

    return {
        "prompt": prompt,
        "preferred": preferred,
        "dispreferred": dispreferred,
        "type": contradiction_type
    }

# ----------------------------
# 4. Generate Dataset
# ----------------------------

def generate_dataset(n_per_type: int = 5) -> List[Dict]:
    dataset = []
    for contradiction_type in CONTRADICTION_TYPES:
        print(f"Generating {n_per_type} pairs for type: {contradiction_type}")
        for _ in range(n_per_type):
            prompt = EXAMPLE_PROMPTS[contradiction_type]
            pair = generate_dpo_pair(prompt, contradiction_type)
            dataset.append(pair)
    return dataset

# ----------------------------
# 5. Save Dataset to Google Drive
# ----------------------------

def save_to_jsonl(data: List[Dict], filename: str = "dpo_contradiction_pairs.jsonl"):
    path = "/content/drive/My Drive/" + filename
    with open(path, 'w') as f:
        for entry in data:
            json.dump(entry, f)
            f.write('\n')
    print(f"✅ Saved {len(data)} DPO pairs to {path}")

# ----------------------------
# 6. Run Everything
# ----------------------------

if __name__ == "__main__":
    dataset = generate_dataset(n_per_type=5)
    save_to_jsonl(dataset)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/3.54G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]